# Semana 5: treinos confirmatórios, codificador congelado, LoRA e ajuste completo, sementes 1 a 5

Treinos confirmatórios de `prereg/preregistro.md` (commit `26cc8b7`, seção 4). Treina o R1 (codificador congelado, lr 5e-4), o R3 (ajuste completo, lr 5e-5) e o R2 (LoRA r = 16, α = 16, lr 5e-4, secundário) com as sementes 1 a 5, na receita da grade (`prereg/piloto_lr.md`) e do piloto de sementes. As rodadas 1 e 2 (R1 e R3, sementes 1 a 5, em 24/10/2026) já estão feitas e registradas: se os arquivos delas estiverem no Drive, o notebook as pula ("já feita") e treina só o R2 (rodada 3).

| O que roda | Detalhe |
|---|---|
| R2, sementes 1 a 5 | 20,6 M parâmetros treináveis (LoRA em q, v, fc1 e fc2, mais pescoço, UperNet e cabeça), cerca de 13 min por execução na T4, com a validação a cada 5 épocas |

**Só a validação é lida.** Teste, Bolívia, WorldFloods v2 e Rio Grande do Sul não são tocados: a inferência nesses conjuntos só acontece depois do script de análise congelado (seção 10 do pré-registro), com códigos de regime sorteados. Abra o Colab logada na conta do mestrado, com GPU T4. Tempo total da rodada 3: cerca de 1 hora e 10 minutos (cinco execuções, a partir do tempo da grade, que foi de 12 min para o R2, mais a validação a cada 5 épocas).

Regras do pré-registro que este notebook aplica:

- Sementes confirmatórias de 1 a 5. As sementes do piloto (0, 101 e 102) nunca são reutilizadas.
- 50 épocas fixas, sem parada antecipada. Vale o modelo da época 50.
- A `val/mIoU` é registrada a cada 5 épocas (precondição 3, seção 8).
- Uma execução falha se a perda ficar não finita ou se a `val/mIoU` da época 50 ficar abaixo de 0,80. A execução que falha é repetida uma vez com a mesma semente. Se falhar de novo, o notebook para e a semente 6 a substitui (você troca na lista `EXECUCOES`). Toda falha fica registrada e é relatada.
- Inferência em 32 bits na validação, com a margem de logit de cada pixel pontuado salva, para o limiar conforme τ e a temperatura.

Etapas:

1. Instalação com versões exatas. Se a célula pedir, reinicie a sessão (Ambiente de execução > Reiniciar sessão) e continue da célula 2.
2. Drive e pastas.
3. Backbone pré-treinado, conferido pelo hash.
4. Chips de treino e de val no disco local, com as listas conferidas contra o repositório.
5. Definições (nada roda aqui).
6. Treino e inferência na val. Se a sessão cair, rode de novo a partir da célula 2: as execuções já feitas são puladas.
7. Resumo das execuções.

Guarde a saída das células 6 e 7. Cada R3 grava cerca de 1,3 GB de pesos no Drive; confira o espaço livre antes.

In [ ]:
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
import os, re, json, time, hashlib, datetime, shutil
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

RES = f'{RAIZ}/resultados/semana5'
os.makedirs(RES, exist_ok=True)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

CHAVE_RX = re.compile(r'([A-Za-z-]+_\d+)')
def chave_de(nome):
    achado = CHAVE_RX.search(os.path.basename(str(nome)))
    assert achado, f'Nome de chip sem chave reconhecível: {nome!r}'
    return achado.group(1)

print(f'Resultados em {RES}')

In [ ]:
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
import urllib.request, concurrent.futures as cf

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
SP = f'{DADOS}/v1.1/splits/flood_handlabeled'
SPLITS_DRIVE = f'{RAIZ}/dados/splits'
for d in (S2, LB, SP, SPLITS_DRIVE):
    os.makedirs(d, exist_ok=True)

SHA_LISTA = {'treino': '4d77d923fcedcf1dca02606ec15acfd2e90238ea06f0b40169b470ca4bd756b5',
             'val': '14a372859fb18b1f7f9791810b7a233c2cd53aaab84f74721f9ac3476ba50504'}

def chaves_csv(nome):
    destino = os.path.join(SPLITS_DRIVE, nome)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BUCKET + 'splits/flood_handlabeled/' + nome, destino)
    chaves = []
    for linha in open(destino).read().strip().splitlines():
        primeiro = os.path.basename(linha.split(',')[0].strip())
        chave = re.sub(r'_(S1Hand|S2Hand|LabelHand)(\.tif)?$', '', primeiro)
        assert chave and chave != primeiro, f'Linha inesperada em {nome}: {linha!r}'
        chaves.append(chave)
    return chaves

TREINO, VAL = chaves_csv('flood_train_data.csv'), chaves_csv('flood_valid_data.csv')
for papel, chaves in (('treino', TREINO), ('val', VAL)):
    lista = '\n'.join(sorted(f'{c}_LabelHand.tif' for c in chaves)) + '\n'
    assert hashlib.sha256(lista.encode()).hexdigest() == SHA_LISTA[papel], (
        f'Lista de {papel} diferente da registrada em splits/README.md.')
assert not set(TREINO) & set(VAL)
VAL_SET = set(VAL)

def baixar(url_e_destino):
    url, destino = url_e_destino
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return 0
    for tentativa in range(4):
        try:
            urllib.request.urlretrieve(url, destino + '.parcial')
            os.replace(destino + '.parcial', destino)
            return 1
        except Exception as erro:
            ultimo = erro
            time.sleep(2 * (tentativa + 1))
    raise RuntimeError(f'Falhou: {url} ({ultimo})')

trabalhos = []
for chave in TREINO + VAL:
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(16) as pool:
    novos = sum(pool.map(baixar, trabalhos))
open(f'{SP}/flood_train_data.txt', 'w').write('\n'.join(TREINO) + '\n')
open(f'{SP}/flood_valid_data.txt', 'w').write('\n'.join(VAL) + '\n')
print(f'{len(trabalhos)} arquivos conferidos, {novos} baixados agora, em {time.perf_counter() - t0:.0f} s. '
      f'Treino {len(TREINO)}, val {len(VAL)}; listas iguais às do repositório.')

## Definições

Receita idêntica à da grade da semana 3 e do piloto de sementes: recorte aleatório de 224 pixels com espelhamentos, lote 16, entropia cruzada com `ignore_index` −1, AdamW com decaimento de peso 0,05, cosseno com `T_max` 50, `head_dropout` 0,1, precisão mista de 16 bits. Só mudam a semente, o regime e a taxa de aprendizado do regime. A validação roda a cada 5 épocas, e a `val/mIoU` de cada uma entra no registro. Depois das 50 épocas, a inferência em 32 bits percorre os 89 chips inteiros da val e grava a margem de logit (água menos terra) de cada pixel pontuado.

In [ ]:
import gc, warnings
import torch
import lightning as L
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from importlib.metadata import version
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
ACELERADOR, PRECISAO, DISPOSITIVO = 'gpu', '16-mixed', 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
LR = {'R1': 5e-4, 'R2': 5e-4, 'R3': 5e-5}
LOTE, EPOCAS, VAL_A_CADA = 16, 50, 5
LIMITE_MIOU, ALFA = 0.80, 0.10
PREREGISTRO_COMMIT = '26cc8b7'
SEMENTES_DO_PILOTO = {0, 101, 102}
TRABALHADORES = min(8, os.cpu_count() or 2)

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE, num_workers=TRABALHADORES, bands=BANDAS,
        train_transform=[A.RandomCrop(height=224, width=224, p=1.0), A.HorizontalFlip(p=0.5),
                         A.VerticalFlip(p=0.5), para_tensor()],
        val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=True, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def tarefa(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    if regime == 'R2':
        args['peft_config'] = LORA
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR[regime], optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=(regime == 'R1'), plot_on_val=0)
    t.configure_models()
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    if regime == 'R2':
        n_lora = sum(1 for n, _ in t.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, f'R2: {n_lora} módulos LoRA, esperado 96.'
    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, f'{regime}: {treinaveis / 1e6:.2f} M treináveis'
    return t

class Vigia(L.Callback):
    """Guarda a val/mIoU a cada validação e para o treino se a perda ficar não finita."""
    def __init__(self):
        self.curva_val_miou = []
        self.perda_nao_finita = None

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        perda = outputs['loss'] if isinstance(outputs, dict) else outputs
        if torch.is_tensor(perda) and not bool(torch.isfinite(perda).all()) and self.perda_nao_finita is None:
            self.perda_nao_finita = dict(epoca=trainer.current_epoch + 1, lote=int(batch_idx))
            trainer.should_stop = True

    def on_validation_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        valor = trainer.callback_metrics.get('val/mIoU')
        if valor is not None:
            self.curva_val_miou.append([trainer.current_epoch + 1, float(valor)])

def inferir_val(modelo):
    dm = datamodule()
    dm.setup('validate')
    ds = dm.val_dataset
    chaves = [chave_de(f) for f in ds.image_files]
    assert chaves == [chave_de(f) for f in ds.segmentation_mask_files], 'Imagens e rótulos da val fora de ordem.'
    assert sorted(chaves) == sorted(VAL), 'Chips da val diferentes da lista registrada.'
    carregador = DataLoader(ds, batch_size=8, shuffle=False, num_workers=TRABALHADORES)
    modelo = modelo.to(DISPOSITIVO).float().eval()
    margens, rotulos, tamanhos = [], [], []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for lote in carregador:
            imagem = dm.aug({'image': lote['image'].to(DISPOSITIVO), 'mask': lote['mask'].to(DISPOSITIVO)})['image']
            assert imagem.dtype == torch.float32 and imagem.ndim == 4 and imagem.shape[1] == len(BANDAS), tuple(imagem.shape)
            z = modelo(imagem).output.float()
            assert tuple(z.shape[-2:]) == tuple(lote['mask'].shape[-2:]), f'Saída {tuple(z.shape)}'
            m = (z[:, 1] - z[:, 0]).cpu().numpy()
            for mi, yi in zip(m, lote['mask'].numpy()):
                pontuado = (yi == 0) | (yi == 1)
                margens.append(mi[pontuado].astype(np.float32))
                rotulos.append(yi[pontuado].astype(np.uint8))
                tamanhos.append(int(pontuado.sum()))
    assert len(tamanhos) == len(chaves)
    return chaves, np.array(tamanhos, dtype=np.int64), np.concatenate(margens), np.concatenate(rotulos)

def miou_binario(pred, y):
    ious = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious))

def caminhos(regime, semente, tentativa=None):
    base = f'{regime}_s{semente}'
    return dict(registro=os.path.join(RES, f'exec_{base}_t{tentativa}.json') if tentativa else None,
                margens=os.path.join(RES, f'margens_{base}.npz'),
                pesos=os.path.join(RES, f'pesos_{base}.pt'))

def execucao_concluida(regime, semente):
    for tentativa in (1, 2):
        p = caminhos(regime, semente, tentativa)
        if os.path.exists(p['registro']) and json.load(open(p['registro'])).get('situacao') == 'ok':
            return os.path.exists(p['margens']) and os.path.exists(p['pesos'])
    return False

import psutil
LIMITE_RAM_PCT = 60   # cuidado operacional entre execuções; não faz parte do método

def ram():
    m = psutil.virtual_memory()
    return (m.total - m.available) / 1e9, m.total / 1e9, m.percent

def executar(regime, semente, tentativa):
    p = caminhos(regime, semente, tentativa)
    L.seed_everything(semente, workers=True)
    t, dm, vigia = tarefa(regime), datamodule(), Vigia()
    trainer = L.Trainer(accelerator=ACELERADOR, devices=1, precision=PRECISAO, max_epochs=EPOCAS,
                        check_val_every_n_epoch=VAL_A_CADA, num_sanity_val_steps=0, logger=False,
                        enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
                        callbacks=[vigia])
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    trainer.fit(t, datamodule=dm)
    minutos = (time.perf_counter() - t0) / 60

    base = dict(regime=regime, semente=semente, tentativa=tentativa, lr=LR[regime], epocas=EPOCAS, lote=LOTE,
                preregistro_commit=PREREGISTRO_COMMIT,
                gerado_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                curva_val_miou=vigia.curva_val_miou, minutos=minutos,
                pico_memoria_gb=torch.cuda.max_memory_allocated() / 1e9, gpu=torch.cuda.get_device_name(0),
                backbone_sha256=SHA_BACKBONE,
                versoes={q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy')})

    motivo = None
    final = vigia.curva_val_miou[-1] if vigia.curva_val_miou else None
    if vigia.perda_nao_finita:
        motivo = f"perda não finita na época {vigia.perda_nao_finita['epoca']}, lote {vigia.perda_nao_finita['lote']}"
    elif final is None or final[0] != EPOCAS:
        motivo = 'sem validação na época 50'
    elif not np.isfinite(final[1]):
        motivo = 'val/mIoU não finita na época 50'
    elif final[1] < LIMITE_MIOU:
        motivo = f'val/mIoU {final[1]:.4f} na época 50, abaixo de {LIMITE_MIOU}'
    if motivo:
        base.update(situacao='falhou', motivo=motivo)
        json.dump(base, open(p['registro'], 'w'), indent=2, ensure_ascii=False)
        print(f'{regime} semente {semente}, tentativa {tentativa}: FALHOU ({motivo}).')
        del t, trainer, dm
        gc.collect(); torch.cuda.empty_cache()
        return base

    chaves, tamanhos, margem, rotulo = inferir_val(t.model)
    pred = (margem > 0).astype(np.uint8)
    miou_32 = miou_binario(pred, rotulo)
    taxa_erro = float((pred != rotulo).mean())
    assert np.isfinite(margem).all(), 'Margens não finitas na inferência em 32 bits: não grave; investigue antes de seguir.'
    np.savez_compressed(p['margens'], chaves=np.array(chaves), tamanhos=tamanhos, margem=margem, rotulo=rotulo)
    estado = t.model.state_dict()
    if regime == 'R1':
        estado = {k: v for k, v in estado.items() if not k.startswith('encoder.')}
    torch.save({k: v.detach().cpu() for k, v in estado.items()}, p['pesos'])

    base.update(situacao='ok', val_miou_epoca_50=final[1], miou_val_32bits=miou_32,
                taxa_erro_val=taxa_erro, alfa_maior_que_taxa_de_erro=bool(ALFA > taxa_erro),
                pixels_pontuados=int(tamanhos.sum()), chips=len(chaves), chips_com_pixel=int((tamanhos > 0).sum()),
                arquivos=dict(margens=p['margens'], pesos=p['pesos']),
                pesos_sem_codificador=(regime == 'R1'))
    json.dump(base, open(p['registro'], 'w'), indent=2, ensure_ascii=False)
    print(f"{regime} semente {semente}, tentativa {tentativa}: val/mIoU na época 50 {final[1]:.4f}, "
          f"mIoU em 32 bits {miou_32:.4f}, erro na val {100 * taxa_erro:.2f}%, {minutos:.1f} min, "
          f"pico {base['pico_memoria_gb']:.1f} GB, {base['pixels_pontuados']:,} pixels em {base['chips_com_pixel']} chips")
    if abs(miou_32 - final[1]) > 0.01:
        print(f'   Atenção: mIoU em 32 bits e val/mIoU do treino diferem em {abs(miou_32 - final[1]):.4f}. Guarde esta saída.')
    if not base['alfa_maior_que_taxa_de_erro']:
        print(f'   Atenção: α = {ALFA} não é maior que a taxa de erro na val (precondição 1). Relatar.')
    del t, trainer, dm, margem, rotulo
    gc.collect(); torch.cuda.empty_cache()
    return base

print('Definições carregadas. Nada foi treinado ainda.')

## Treino e inferência na validação

Uma execução por vez, na ordem da lista. Antes de cada execução o notebook mostra o uso de memória RAM e, se passar de 60%, para e pede para reiniciar a sessão (a memória acumula entre execuções no Colab). Nada se perde: ao rodar de novo, o que está feito é pulado. Se uma execução falhar, ela é repetida uma vez com a mesma semente. Se falhar de novo, o notebook para com a instrução da regra de substituição.

In [ ]:
EXECUCOES = [('R1', 1), ('R3', 1), ('R1', 2), ('R3', 2),
              ('R1', 3), ('R3', 3), ('R1', 4), ('R3', 4), ('R1', 5), ('R3', 5),
              ('R2', 1), ('R2', 2), ('R2', 3), ('R2', 4), ('R2', 5)]

for regime, semente in EXECUCOES:
    assert regime in LR, f'Regime {regime} não é treinado neste notebook.'
    assert semente not in SEMENTES_DO_PILOTO and semente >= 1, f'Semente {semente} é do piloto ou inválida.'
    if execucao_concluida(regime, semente):
        print(f'{regime} semente {semente}: já feita.')
        continue
    usado, total, pct = ram()
    print(f'RAM antes de {regime} semente {semente}: {usado:.1f} de {total:.1f} GB ({pct:.0f}%).')
    if pct > LIMITE_RAM_PCT:
        print(f'PAREI aqui para a memória não acabar no meio de um treino. Reinicie a sessão (Ambiente de execução > '
              f'Reiniciar sessão) e rode da célula 2 em diante: o notebook pula o que está feito e retoma em '
              f'{regime} semente {semente}.')
        break
    for tentativa in (1, 2):
        registro = caminhos(regime, semente, tentativa)['registro']
        if os.path.exists(registro) and json.load(open(registro)).get('situacao') == 'falhou':
            print(f'{regime} semente {semente}, tentativa {tentativa}: já registrada como falha.')
            continue
        if executar(regime, semente, tentativa)['situacao'] == 'ok':
            break
    else:
        raise RuntimeError(
            f'{regime} semente {semente} falhou duas vezes. Pela seção 4 do pré-registro, a semente 6 a substitui '
            f'(depois a 7, e assim por diante). Troque ({regime!r}, {semente}) por ({regime!r}, 6) na lista EXECUCOES '
            'e rode esta célula de novo. As duas falhas ficam registradas e são relatadas.')

print('\nGuarde a saída desta célula.')

## Resumo

In [ ]:
import glob
registros = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(RES, 'exec_*_t*.json')))]
registros.sort(key=lambda r: (r['regime'], r['semente'], r['tentativa']))
print(f"{'regime':6s} {'semente':7s} {'tent.':5s} {'situação':8s} {'val/mIoU 50':11s} {'mIoU 32 bits':12s} {'erro val %':10s} {'min':5s}")
for r in registros:
    if r['situacao'] == 'ok':
        print(f"{r['regime']:6s} {r['semente']:<7d} {r['tentativa']:<5d} {'ok':8s} {r['val_miou_epoca_50']:<11.4f} "
              f"{r['miou_val_32bits']:<12.4f} {100 * r['taxa_erro_val']:<10.2f} {r['minutos']:<5.1f}")
    else:
        print(f"{r['regime']:6s} {r['semente']:<7d} {r['tentativa']:<5d} {'FALHOU':8s} {r['motivo']}")

print('\nCurva da val/mIoU a cada 5 épocas (execuções ok):')
for r in registros:
    if r['situacao'] == 'ok':
        print(f"  {r['regime']} s{r['semente']}: " + ' '.join(f'{v:.3f}' for _, v in r['curva_val_miou']))

falhas = [r for r in registros if r['situacao'] == 'falhou']
print(f'\nFalhas registradas: {len(falhas)}.')
print('α maior que a taxa de erro na val em todas as execuções ok:',
      all(r['alfa_maior_que_taxa_de_erro'] for r in registros if r['situacao'] == 'ok'))
print('\nGuarde a saída desta célula.')